In [ ]:
import csv
from datetime import datetime
import pandas as pd 
import numpy as np
import re
import random
import matplotlib.pyplot as plt

In [ ]:
#importing the csv file
file_path = r'data_collection\data_collection\tayara.tn\tayara_tn.csv'
full_df = pd.read_csv(file_path)
full_df.head()

In [ ]:
#typo in previous column title name
full_df = full_df.rename(columns={' title':'title'})
#dropping rows that promote cars renting ads
mask = (
    full_df['title'].str.contains('location|كرا|louer', case=False, na=False) |
    full_df['description'].str.contains('location|كرا|louer', case=False, na=False)
)
full_df = full_df[~mask]

In [ ]:
#dropping unnessary columns; description, interior, ownership, title, and link
df = full_df.drop(columns=['description','interior','link', 'ownership','title'])

In [ ]:
df.dtypes

In [ ]:
#checking null values per column
null = df.isnull()
null.apply(pd.value_counts).fillna(0)

In [ ]:
#converting columns datatypes, starting with mileage, enginesize, fiscalpower
df = df.astype('str')

def extract_number_regex(text):
    try:
        #keep only digits and decimal points
        cleaned = re.sub(r'[^0-9.]', '', str(text))
        return float(cleaned) if cleaned else None
    except:
        return None
for col in ['mileage', 'engine-size', 'fiscal-power']:
    df[col] = df[col].map(extract_number_regex)


In [ ]:
#date data was collected is the 10th of July
'''
3 months ago = 10th april
2 months ago = 10th mai
1 month ago = 10th june
27 days ago to 11 days ago = 10th june
10 days ago and less = random dates between 1st July and 10th July
'''
publish_dates = df[['publish-date']].value_counts().index.tolist()

In [ ]:
#transforming publish-dates to datetime datatypes: (chatgpt help)
from datetime import datetime, timedelta
import re

collection_date = datetime(2025, 7, 10)

def date_transformation(text):
    text = text.strip()
    
    # handle 'a day/month ago'
    if text.startswith('a '):
        text = '1 ' + text[2:]
    
    # extract number and unit
    match = re.match(r'(\d+)\s+(day|month|hour)s?\s+ago', text)
    if match:
        value, unit = int(match.group(1)), match.group(2)
        if unit == 'month':
            month = collection_date.month - value
            year = collection_date.year
            while month <= 0:
                month += 12
                year -= 1
            return datetime(year, month, collection_date.day)
        elif unit == 'day':
            return (collection_date - timedelta(days=value)).replace(hour=0, minute=0, second=0, microsecond=0)
        elif unit == 'hour':
            days_back = 1 if value >= collection_date.hour else 0
            return (collection_date - timedelta(days=days_back)).replace(hour=0, minute=0, second=0, microsecond=0)
    
    return collection_date  # fallback

df['publish-date'] = df['publish-date'].map(date_transformation)


In [ ]:
#converting the price datatype to an integer
df['price'] = df['price'].astype('float').astype('int')
df.head()

In [ ]:
#transforming irregular prices to the original (eg: 47 to 47000)
weird_prices = [111,222,333,444,555,666,777,888,999,1111,2222,3333,4444,5555,6666,7777,8888,9999,123,1234]

def price_transformation(price):
    if price in weird_prices:
        return 0
    if price < 500:
        price = price*1000
    return price
df['price'] = df['price'].apply(price_transformation)

#setting sequential fake numbers to 0, eg: 111111
def is_repeated(price):
    s = str(int(float(price)))
    return bool(re.fullmatch(r"(\d)\1+", s))
df.loc[df['price'].apply(is_repeated), 'price'] = 0

In [ ]:
#we handle NaN values for integer transformation, we set NaN values to 999999
df['mileage'] = df['mileage'].replace([np.inf, -np.inf], np.nan).fillna(999999).astype(int)

df['mileage'] = df['mileage'].astype('int')

In [ ]:
#dropping negative mileage values
df = df[df['mileage']>=0]
#dropping mileage outliers
df = df[df['mileage']<500000]
#transforming weird mileage values to the normal format
def mileage_transformation(mileage):
    if mileage<400:
        mileage *= 1000
    return mileage
df['mileage'] = df['mileage'].apply(mileage_transformation)
df = df[df['mileage']>=5000]

In [ ]:
#setting nan strings to NaN numpy values
columns = df.columns.values.tolist()
for column in columns:
    df[column] = df[column].replace('nan', np.nan)
df['price'].replace(0, np.nan, inplace=True)

In [ ]:
#dealing with circulation date column, dropping misleading rows, and dealing with wrong formats
df = df[df['circulation-date'].astype(str).str.fullmatch(r'\d{4}')]
df['circulation-date'] = df['circulation-date'].replace('nan', np.nan).fillna(0).astype('float').astype('int')
df = df[(df['circulation-date']<2025) & (df['circulation-date']>1978)]

In [ ]:
#transforming engine-size to 1000cc instead of X/L unit

df['engine-size'] = df['engine-size'].map(lambda x: x*1000)
df['engine-size'] = df['engine-size'].astype('Int64')

In [ ]:
#setting eletric cars engine-size to 0
df.loc[df['fuel']=='Electrique', 'engine-size'] = 0

In [ ]:
#setting high fiscal-power values to NaN
df.loc[df['fiscal-power']>=35, 'fiscal-power'] = np.nan

#transforming fiscal-power dtype to integer
df['fiscal-power'] = df['fiscal-power'].round().astype('Int64')

In [ ]:
#replacing 'Autres' body-type values, because of their insignificance
df['body-type'].replace('Autres', np.nan, inplace=True)

In [ ]:
df['body-type'].value_counts().index.tolist()

In [ ]:
#dropping rows with 'autres' brand & model
df = df[df['model'] != 'Autres']
#dropping null model and brand car rows
df = df.dropna(subset=['brand','model'])

In [ ]:
#setting a new column 'car-age' in months
def generate_random_date_suffix(): #chatgpt help, to simulate real world car age
    month = random.randint(1, 12)
    if month in [1, 3, 5, 7, 8, 10, 12]:
        day = random.randint(1, 31)
    elif month in [4, 6, 9, 11]:
        day = random.randint(1, 30)
    else:  # February
        day = random.randint(1, 28)  # Not handling leap years for simplicity
    
    return f"-{month:02d}-{day:02d}"

# Apply random dates to each circulation year
df['circulation-date'] = df['circulation-date'].astype(str).apply(
    lambda x: pd.to_datetime(x + generate_random_date_suffix())
)
df['publish-date'] = pd.to_datetime(df['publish-date'])

avg_publish_date = df['publish-date'].mean().date()
df['car-age'] = df['circulation-date'].apply(lambda x: (avg_publish_date - x.date()).days //30.42 if pd.notna(x) else np.nan)
df['car-age'] = df['car-age'].astype('int')

In [ ]:
#filling NaN values
#fuel:
df['fuel'] = df.groupby(['brand', 'model'])['fuel'].transform(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan))
#gear
df['gear'] = df.groupby(['brand', 'model'])['gear'].transform(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan))
#engine-size
df['engine-size'] = df.groupby(['brand','model','fuel','gear'])['engine-size'].transform(
    lambda x: x.fillna(x.median()))
#body-type
df['body-type'] = df.groupby(['brand', 'model'])['body-type'].transform(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan))
#dropping NaN rows:
df = df.dropna(subset=['body-type','engine-size','fuel','gear'])

In [ ]:
#dropping price outliers
df = df.drop(df[(df['brand'].isin(['Renault','Peugeot'])) & (df['car-age'].isin([421, 507])) & (df['price'].isin([200000,100000]))].index)

In [ ]:
#dealing with price outliers, chatgpt help
brands_to_keep = ['Land Rover','Jeep','BMW','Porsche','Mercedes-Benz','Toyota','Cadillac']

df = df.drop(
    df[
        ((~df['brand'].isin(brands_to_keep)) & (df['car-age'] < 120) & (df['price'] >= 250000))
        | ((df['car-age'] >= 120) & (df['price'] >= 250000))
    ].index
)
#setting prices under 7000tnd to NaN
df.loc[df['price']<=7000, 'price'] = np.nan
#drop automatic cars that are less than 30000
df = df.drop(df[(df['gear'] == 'Automatique') & (df['price']<30000)].index)
#drop new cars that are less than 30000
df = df.drop(df[(df['car-age'] < 36) & (df['price'] < 30000)].index)

In [ ]:
#identifying mileage outliers
import matplotlib.pyplot as plt

plt.scatter(df['car-age'], df['mileage'], alpha=0.5)
plt.xlabel('Age (months)')
plt.ylabel('Mileage (km)')
plt.title('Mileage vs Age')
plt.show()


In [ ]:
#dropping mileage outliers
df = df.drop(df[(df['car-age'] > 400) & (df['mileage'] < 300000)].index)


In [ ]:
#dropping duplicates
df = df.drop_duplicates(subset=['model','brand','mileage','price'])

In [ ]:
#making the whole content of the dataframe lower case
df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

In [ ]:
#standardization process, column by column
df.columns

In [ ]:
dfauto = pd.read_csv(r'data_wrangling\csv\automobiletn_set.csv')
dfauto.columns

In [ ]:
#converting datatypes
df['engine-size'] = df['engine-size'].astype('float64')
df['car-age'] = df['car-age'].astype('float64')

In [ ]:
#dealing with the brand column
#tunisieautomobile has 53 brands listed
#tayaratn has 48 brands listed

brand_map = {
    'citroen': 'citroën',    
    'ac': 'ac',                
    'foton': 'foton',            
    'infiniti': 'infiniti',                              
    'wallyscar': 'wallyscar',
    'rover': 'land rover',
    'alfa romeo': 'alfa romeo',
    'audi': 'audi',
    'bmw': 'bmw',
    'byd': 'byd',
    'chery': 'chery',
    'chevrolet': 'chevrolet',
    'dacia': 'dacia',
    'dodge': 'dodge',
    'dongfeng': 'dongfeng',
    'fiat': 'fiat',
    'ford': 'ford',
    'geely': 'geely',
    'haval': 'haval',
    'honda': 'honda',
    'hummer': 'hummer',
    'hyundai': 'hyundai',
    'isuzu': 'isuzu',
    'iveco': 'iveco',
    'jaguar': 'jaguar',
    'jeep': 'jeep',
    'kia': 'kia',
    'lancia': 'lancia',
    'land rover': 'land rover',
    'mahindra': 'mahindra',
    'mazda': 'mazda',
    'mercedes-benz': 'mercedes-benz',
    'mg': 'mg',
    'mini': 'mini',
    'mitsubishi': 'mitsubishi',
    'nissan': 'nissan',
    'opel': 'opel',
    'peugeot': 'peugeot',
    'porsche': 'porsche',
    'renault': 'renault',
    'seat': 'seat',
    'skoda': 'skoda',
    'smart': 'smart',
    'ssangyong': 'ssangyong',
    'suzuki': 'suzuki',
    'toyota': 'toyota',
    'volkswagen': 'volkswagen',
    'volvo': 'volvo',
}

df['brand'] = df['brand'].str.lower().replace(brand_map)

In [ ]:
#dealing with the model column
model_mapping = {
    'classe gle': 'gle',
    'classe glc': 'glc',
    'classe c coupe': 'classe c coupé',
    'classe a': 'classe a',
    'classe s': 'classe s',
    'classe e': 'classe e',
    'classe c': 'classe c',
    'classe cla': 'cla',
    'classe ml': 'ml',
    'classe gla': 'gla',
    'classe cls': 'cls',
    'classe gl': 'classe gl',
    'classe m': 'ml',
    'classe cl': 'cl',
    'classe b': 'classe b',
    'classe sls': 'classe sls',
    'classe v': 'classe v',
    'classe x': 'classe x',
    'classe g': 'classe g',
    'serie 1': 'série 1',
    'serie 2': 'série 2 coupé',
    'serie 3': 'série 3',
    'serie 3 gt': 'série 3 coupé',
    'serie 4': 'série 4 coupé',
    'serie 5': 'série 5',
    'i 10': 'grand i10', 
    'i 20': 'i20',
    'i 30': 'i30',
    'grand i10': 'grand i10',
    'range rover': 'range rover',
    'range rover evoque': 'range rover evoque',
    'range rover sport': 'range rover sport',
    'range rover velar': 'range rover velar',
    'rav 4': 'rav 4',
    'c-max': 'c-max',  
    'focus c-max': 'focus',  
    'megane estate': 'megane',
    'golf 4': 'golf 4',  
    'golf 5': 'golf 5',
    'golf 6': 'golf 6',
    'golf 7': 'golf 7',
    'golf 8': 'golf 8',
    'passat cc': 'passat cc',
    'leon st': 'leon sc',  
    'grand scenic': 'grand scenic',  
    'logan mcv': 'logan mcv',
    'c-elysée': 'c-elysée',
    'c4 cactus': 'c4 cactus',
    'c4 aircross': 'c4 aircross',  
    'c4 picasso': 'c4 picasso',  
    'grand c4 picasso': 'grand c4 picasso', 
    '308 sw': '308',  
    '407 sw': '407', 
    '207 sw': '207',
    'ix 35': 'ix35',
    'x-trail': 'x-trail',
    'hr-v': 'hr-v',
    'rio': 'rio', 
    '500': '500',
    'tiggo': 'tiggo 8',
    'focus': 'focus',
    'palio': 'palio',
    'polo': 'polo',
    'partner': 'partner',
    'picanto': 'picanto',
    'c3': 'c3',
    'c4': 'c4',
    'c1': 'c1', 
    'tivoli': 'tivoli',
    'octavia': 'octavia',
    'accent': 'accent',
    'megane': 'megane',
    'leon': 'leon',
    '208': '208',
    'kona': 'kona',
    'corsa': 'corsa',
    'sportage': 'sportage',
    'kadjar': 'kadjar',
    'cx-5': 'cx-5',
    'ecosport': 'ecosport',
    'q5': 'q5',
    'scenic': 'scenic', 
    'mito': 'mito',
    'clio': 'clio',
    'elantra': 'elantra',
    'dokker': 'dokker',  
    '206': '206',
    't-roc': 't-roc',
    'yaris': 'yaris',
    'tucson': 'tucson',
    'a4': 'a4',
    'capture': 'captur',  
    'qashqai': 'qashqai',
    'd-max': 'd-max',  
    'prado': 'prado',
    'celerio': 'celerio',
    'cayenne': 'cayenne',
    'swift': 'swift',
    'sandero': 'sandero',  
    '2008': '2008',
    'bipper': 'bipper',
    'juke': 'juke',
    'a3': 'a3',
    'punto': 'punto evo', 
    'fluence': 'fluence', 
    'l200': 'l200 double cabine',  
    'berlingo': 'berlingo',
    'caddy': 'caddy',
    '308': '308',
    'ranger': 'ranger',
    'ibiza': 'ibiza',
    'connect': 'connect',  
    'jolion': 'jolion',
    'astra': 'astra',
    'grande punto': 'grande punto',
    '3008': '3008',
    'bt-50': 'bt-50',
    'passat': 'passat',
    'qq': 'qq',
    '500x': '500x',
    'micra': 'micra',
    'cc': 'cc',
    '106': '106',
    'x5': 'x5',
    'fiesta': 'fiesta',
    'ka': 'ka',
    '5008': '5008',  
    'logan': 'logan',  
    'nemo': 'nemo',
    'vito': 'vito',  
    'hilux': 'hilux', 
    'q3': 'q3',
    'ceed': "ceed",  
    'symbol': 'symbol',
    'aygo': 'aygo',
    '407': '407',
    'cooper': 'cooper', 
    'land cruiser': 'land cruiser',
    '3': '3',
    'zs': 'zs',
    'ypsilon': 'ypsilon',
    'fabia': 'fabia',
    'h6': 'h6',
    'c5': 'c5',
    'fiorino': 'fiorino',
    'panda': 'panda',
    'jetta': 'jetta',
    '301': '301',
    'creta': 'creta',
    'transit': 'transit',  
    'jumpy': 'jumpy combi',  
    'wrangler': 'wrangler',
    'tiguan': 'tiguan',
    '504': '504',
    'cerato': 'cerato 5p',  
    'c-hr': 'c-hr',
    'tipo': 'tipo 5 portes', 
    '206+': '206+',  
    'giulietta': 'giulietta',
    'amarok': 'amarok',
    'pajero sport': 'pajero sport',
    'panamera': 'panamera',
    'duster': 'duster',
    'cherokee': 'cherokee',
    'corolla': 'corolla',
    'combo': 'combo cargo', 
    'captiva': 'captiva',  
    'cruze': 'cruze',
    'vitara': 'vitara',
    'navara': 'navara',
    'x-trail': 'x-trail',
    'x1': 'x1',
    'compass': 'compass',
    'twingo': 'twingo',  
    'patrol gr': 'patrol',  
    '5': '5',
    'boxer': 'boxer',
    'sonata': 'sonata',  
    'xf': 'xf',
    'smart': 'smart',  
    'kangoo': 'kangoo',  
    'cayman': 'cayman',  
    'laguna': 'laguna',  
    'a1': 'a1 sportback',  
    '306': '306', 
    'jumper': 'jumper',
    'x6': 'x6',
    'fusion': 'fusion',
    'ds3': 'ds3',
    'city': 'city',
    'touareg': 'touareg',
    'x3': 'x3',
    '207': '207',
    'kuga': 'kuga',
    'uno': 'uno', 
    'mondeo': 'mondeo', 
    'ducato': 'ducato',
    'accord': 'accord',
    'sprinter': 'sprinter van', 
    'a6': 'a6',
    'a5': 'a5',  
    'spark': 'spark',  
    'scirocco': 'scirocco',  
    'rapid': 'rapid',  
    'new beetle': 'new beetle',
    'patrol': 'patrol',
    'santa fe': 'santa fe', 
    'a8': 'a8 l',  
    'h3': 'h3',
    'daily': 'daily simple cabine', 
    'outlander': 'outlander',  
    'auris': 'auris',  
    'grand cherokee': 'grand cherokee',
    'fox': 'fox',  
    'sorento': 'sorento',
    'jimny': 'jimny 3 portes',
    'h2': 'h2',
    'civic': 'civic',
    'prius': 'prius',  
    'optima': 'optima', 
    'f150': 'f150',  
    '2': '2',
    '6': '6',
    'm': 'm',
    'm3': 'm3',  
    'm4': 'm4',
    'note': 'note',
    'one': 'one',  
    'pajero': 'pajero',
    'siena': 'siena',
    'x4': 'x4',
    'f3': 'f3',
    'hs': 'hs',
    'e3': 'e3',
    'sx3': 'sx3',
    'ax4': 'ax4',
    's50': 's50',
    'k2700': 'k2700',
    'k02': 'k02',
    '508': '508',
    '108': '108'
}

df['model'] = df['model'].str.lower().replace(model_mapping)

In [ ]:
#standardizing the location column
location_mapping = {
    'ben arous ': 'ben arous',
    'sousse ': 'sousse', 
    'ariana ': 'ariana',
    'médenine ': 'médenine',
    'tunis ': 'tunis',
    'gafsa ': 'gafsa',
    'nabeul ': 'nabeul',
    'sfax ': 'sfax',
    'monastir ': 'monastir',
    'béja ': 'béja',
    'tozeur ': 'tozeur',
    'la manouba ': 'la manouba',
    'zaghouan ': 'zaghouan',
    'jendouba ': 'jendouba',
    'gabès ': 'gabès',
    'kébili ': 'kébili',
    'sidi bouzid ': 'sidi bouzid',
    'bizerte ': 'bizerte',
    'mahdia ': 'mahdia',
    'le kef ': 'le kef',
    'kairouan ': 'kairouan',
    'kasserine ': 'kasserine',
    'siliana ': 'siliana',
    'tataouine ': 'tataouine'
}

df['location'] = df['location'].str.lower().replace(location_mapping)

In [ ]:
#exporting the standardized csv file
df.to_csv('tayara_tn_standardized.csv')